In [13]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import konlpy
from konlpy.tag import Twitter
from collections import Counter
import pickle as pickle
from tqdm import tnrange
from tqdm import tqdm

## 1. 기업별 빈도분석

In [6]:
essay_df = pd.read_pickle('essay_df.pkl')

In [7]:
essay_df.head()

,company_id,company,year,season,experience,position,score,feedback,content,views
0,0,농협은행(주),2020,상반기,신입,사무·총무·법무,2,전반적으로 호감을 주지 못하는 내용 성실히 쓰려고 노력한 점 질문의 의도에 맞지 ...,직장은 생계를 꾸려나가는 수단이기도 하지만 저의 가치관을 실현시킬 수 있는 곳이라고...,"15,387"
1,0,농협은행(주),2019,하반기,신입,금융·보험영업,2,지원자만의 강점이 없음 성실히 쓰려고 노력한 점 금융이나 회사 관련 어떤 준비와 ...,농협은 임직원들이 '농심'. '현장'. '공감'이라는 3대 핵심가치를 기반으로 사고...,"7,360"
2,0,농협은행(주),2019,하반기,신입,금융·보험영업,2,"회사, 직무와 관련하여 지원자만의 강점이 없음 성실히 쓰려고 노력한 점 금융이나 ...","""정확한 의도파악을 통한 문제해결"" 저는 정확한 의도 파악을 통해 윗사람과의 원활한...","4,788"
3,0,농협은행(주),2019,상반기,신입,금융·보험영업,3,지속적으로 금융과 농업에 대해 관심을 가진 것 같아 보이지 않음 하나의 관련 경험...,"""농협, 농업과 더욱 가까이"" 4개월 동안 농협은행 취업아카데미와 농협은행 지역본부...","6,845"
4,0,농협은행(주),2019,하반기,신입,금융·보험영업,2,지원자만의 강점이 없음 성실히 쓰려고 노력한 점 금융이나 회사 관련 어떤 준비와 ...,"""새로운 채널 도전"" 1인 미디어가 주목받는 시대에 새로운 채널에 대한 파악이 늦다...","3,851"


In [4]:
#한국어 불용어
stopwords = pd.read_json('stopwords-ko.json')
stopwords_list = list(stopwords[0])
stopwords_list

['!',
 '"',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '...',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ';',
 '<',
 '=',
 '>',
 '?',
 '@',
 '\\',
 '^',
 '_',
 '`',
 '|',
 '~',
 '·',
 '—',
 '——',
 '‘',
 '’',
 '“',
 '”',
 '…',
 '、',
 '。',
 '〈',
 '〉',
 '《',
 '》',
 '가',
 '가까스로',
 '가령',
 '각',
 '각각',
 '각자',
 '각종',
 '갖고말하자면',
 '같다',
 '같이',
 '개의치않고',
 '거니와',
 '거바',
 '거의',
 '것',
 '것과 같이',
 '것들',
 '게다가',
 '게우다',
 '겨우',
 '견지에서',
 '결과에 이르다',
 '결국',
 '결론을 낼 수 있다',
 '겸사겸사',
 '고려하면',
 '고로',
 '곧',
 '공동으로',
 '과',
 '과연',
 '관계가 있다',
 '관계없이',
 '관련이 있다',
 '관하여',
 '관한',
 '관해서는',
 '구',
 '구체적으로',
 '구토하다',
 '그',
 '그들',
 '그때',
 '그래',
 '그래도',
 '그래서',
 '그러나',
 '그러니',
 '그러니까',
 '그러면',
 '그러므로',
 '그러한즉',
 '그런 까닭에',
 '그런데',
 '그런즉',
 '그럼',
 '그럼에도 불구하고',
 '그렇게 함으로써',
 '그렇지',
 '그렇지 않다면',
 '그렇지 않으면',
 '그렇지만',
 '그렇지않으면',
 '그리고',
 '그리하여',
 '그만이다',
 '그에 따르는',
 '그위에',
 '그저',
 '그중에서',
 '그치지 않다',
 '근거로',
 '근거하여',
 '기대여',
 '기점으로',
 '기준으로',
 '기타',
 '까닭으로',
 '까악',
 '까지',
 '까지 미치다',
 '까지도

In [5]:
#contents(자소서 내용) 을 넣었을 때 빈도분

common_del_list = ['되어다','때문','그리고','좋다','자다','없다','같다','싶다','보다', '하다', '있다', '되다', '이다', '돼다', '않다', '그렇다', '아니다', '이렇다', '그렇다', '어떻다']
del_list = stopwords_list + common_del_list
counter_list = []

def counter(contents): 
    
    twitter = Twitter()
    raw_pos_tagged = twitter.pos(contents, norm=True, stem=True) 
 
    word_cleaned = []
    
    for word, tag in raw_pos_tagged: 
        if not tag in ["Josa", "Eomi", "Punctuation", "Foreign"]: 
            if (len(word) != 1) & (word not in del_list): 
                word_cleaned.append(word)

    word_counted = Counter(word_cleaned)
    word_dic = dict(word_counted)
    
    counter_list.append(word_dic)
    return word_dic

#     'word_dic' column에 dict 형식으로 입력
#     df['word_dic'][idx[0]] = dict(sorted_word_dic)
#     return sorted_word_dic
#     print(word_dic)

In [9]:
content_list = []

for x in range (11):
    contents =''
    for i in range(len(essay_df.content[essay_df['company_id']==x])):
        contents += essay_df.content[essay_df['company_id']==x].values[i]
    content_list.append(contents)

In [11]:
with open('content_list.pkl', 'wb') as f:
    pickle.dump(content_list, f)

In [15]:
for x in range (11):
    contents =''
    for i in tnrange(len(essay_df.content[essay_df['company_id']==x])):
        contents += essay_df.content[essay_df['company_id']==x].values[i]
    counter(contents)

/Users/haeunkang/anaconda3/lib/python3.7/site-packages/konlpy/tag/_okt.py:16: UserWarning: "Twitter" has changed to "Okt" since KoNLPy v0.4.5.
  warn('"Twitter" has changed to "Okt" since KoNLPy v0.4.5.')


In [24]:
sorted_counter_list = []

for di in counter_list:
    sorted_counter = sorted(di.items(), key=lambda x:x[1], reverse=True)
    sorted_counter_list.append(sorted_counter)

In [26]:
with open('counter_list.pkl', 'wb') as f:
    pickle.dump(counter_list, f)
    
with open('sorted_counter_list.pkl', 'wb') as f:
    pickle.dump(sorted_counter_list, f)

In [27]:
with open('company_code_dict.pkl', 'rb') as f:
    company_code_dict = pickle.load(f)

In [30]:
for i in range(11):
    print(company_code_dict[i], '의 빈도사전입니다:', '\n')
    print(sorted_counter_list[i][:50], '\n')

농협 의 빈도사전입니다: 

[('농협', 553), ('생각', 466), ('고객', 448), ('통해', 330), ('위해', 287), ('은행', 231), ('경험', 226), ('업무', 210), ('노력', 187), ('많다', 183), ('목표', 170), ('금융', 166), ('받다', 165), ('대한', 150), ('사람', 148), ('이르다', 140), ('만들다', 139), ('농업인', 135), ('발전', 131), ('사회', 129), ('농업', 124), ('서비스', 123), ('상품', 123), ('가장', 119), ('IT', 117), ('활동', 116), ('NH', 116), ('지역', 113), ('이러하다', 113), ('필요하다', 112), ('농민', 111), ('상황', 109), ('새롭다', 108), ('문제', 106), ('결과', 106), ('사업', 104), ('바탕', 104), ('진행', 103), ('중요하다', 99), ('크다', 99), ('다양하다', 98), ('지원', 97), ('소통', 97), ('가지', 96), ('얻다', 93), ('신뢰', 92), ('제공', 92), ('가치', 88), ('오다', 88), ('분석', 88)] 

롯데 의 빈도사전입니다: 

[('고객', 716), ('생각', 605), ('경험', 586), ('업무', 562), ('통해', 546), ('롯데', 533), ('위해', 504), ('영업', 419), ('사람', 384), ('관리', 373), ('노력', 321), ('배우다', 313), ('대한', 312), ('많다', 300), ('만들다', 269), ('받다', 255), ('활동', 249), ('가지', 240), ('이르다', 240), ('다양하다', 232), ('목표', 232), ('결과', 228), ('분석', 223), ('입사', 20

## 2. TF-IDF

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
vectorizer = TfidfVectorizer()
# corpus = [doc1, doc2]
X = vectorizer.fit_transform(corpus).todense()

NameError: name 'corpus' is not defined